# Day 6: Reinforcement Layers - Material Integration

**Phase 2: cs_design, Step 2 - Reinforcement Layers**

This notebook demonstrates the reinforcement layer classes that integrate with steel material models from Phase 1.

## Objectives

1. **ReinforcementLayer**: Single layer with position, area, and material
2. **ReinforcementLayout**: Collection of layers with analysis methods
3. **Material Integration**: Connect to SteelReinforcement from Phase 1
4. **Force Computation**: Calculate N and M for given strains
5. **Helper Functions**: Create common reinforcement patterns

## Architecture

```
ReinforcementLayer (BMCSModel)
├── z: Distance from top
├── A_s: Steel area
├── material: SteelReinforcement
├── get_sig(eps) → stress
├── get_force(eps) → axial force
└── get_moment(eps, y_ref) → moment

ReinforcementLayout
├── layers: List[ReinforcementLayer]
├── add_layer() → add reinforcement
├── get_N_M(kappa, eps_top) → total forces
└── Helper methods for layout management
```

## Integration with Phase 1

This phase connects cs_design with matmod:
- Uses `SteelReinforcement` material model
- Uses `create_steel()` factory for standard grades
- Delegates stress computation to material model

In [ ]:
# Setup
import sys
sys.path.insert(0, '/home/rch/Coding/bmcs_cross_section')

import numpy as np
import matplotlib.pyplot as plt

from bmcs_cross_section.cs_design.reinforcement import (
    ReinforcementLayer,
    ReinforcementLayout,
    create_symmetric_reinforcement,
    create_distributed_reinforcement
)
from bmcs_cross_section.matmod.steel_reinforcement import create_steel

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

## Test 1: ReinforcementLayer - Basic Properties

Create a single reinforcement layer and check integration with material model.

In [ ]:
# Test 1: Single layer
print("=" * 70)
print("TEST 1: ReinforcementLayer - Basic Properties")
print("=" * 70)

# Create layer with default material (B500B)
layer1 = ReinforcementLayer(z=50, A_s=1000)

print(f"\n1. Layer with default material:")
print(f"   Position (z):      {layer1.z:.1f} mm")
print(f"   Area (A_s):        {layer1.A_s:.0f} mm²")
print(f"   Material:          {type(layer1.material).__name__}")
print(f"   Yield strength:    {layer1.f_sy:.1f} MPa")
print(f"   Tensile strength:  {layer1.f_st:.1f} MPa")
print(f"   Elastic modulus:   {layer1.E_s:,.0f} MPa")
print(f"   Yield strain:      {layer1.eps_sy:.5f}")

# Create layer with custom material
steel_b500c = create_steel('B500C')
layer2 = ReinforcementLayer(z=450, A_s=2000, material=steel_b500c, name="Bottom")

print(f"\n2. Layer with B500C material:")
print(f"   Name:              {layer2.name}")
print(f"   Position (z):      {layer2.z:.1f} mm")
print(f"   Area (A_s):        {layer2.A_s:.0f} mm²")
print(f"   Yield strength:    {layer2.f_sy:.1f} MPa")
print(f"   Ductility ratio k: {layer2.material.ductility_ratio:.3f}")

print(f"\n✓ Layer properties correctly inherited from material model!")

## Test 2: Stress-Strain Response

Test that stress computation delegates correctly to material model.

In [ ]:
# Test 2: Stress computation
print("\n" + "=" * 70)
print("TEST 2: Stress-Strain Response")
print("=" * 70)

layer = ReinforcementLayer(z=50, A_s=1000)

# Test various strain levels
strains = np.array([-0.01, -0.005, -0.0025, 0.0, 0.0025, 0.005, 0.01, 0.02, 0.05])
stresses = layer.get_sig(strains)

print(f"\n1. Stress at various strain levels:")
print(f"   {'Strain [-]':<15} {'Stress [MPa]':<15} {'State'}")
print(f"   {'-'*50}")

for eps, sig in zip(strains, stresses):
    if abs(eps) < layer.eps_sy:
        state = "Elastic"
    elif abs(eps) < 0.05:
        state = "Plastic"
    else:
        state = "Hardening"
    print(f"   {eps:<15.5f} {sig:<15.1f} {state}")

# Verify elastic range
eps_elastic = 0.001
sig_elastic = layer.get_sig(eps_elastic)
sig_expected = eps_elastic * layer.E_s

print(f"\n2. Elastic range verification:")
print(f"   ε = {eps_elastic:.5f}")
print(f"   σ = {sig_elastic:.1f} MPa (computed)")
print(f"   σ = {sig_expected:.1f} MPa (expected: ε×E_s)")
print(f"   Match: {np.isclose(sig_elastic, sig_expected)}")

print(f"\n✓ Stress computation working correctly!")

## Test 3: Force and Moment Computation

Test computation of forces and moments for given strains.

In [ ]:
# Test 3: Force and moment
print("\n" + "=" * 70)
print("TEST 3: Force and Moment Computation")
print("=" * 70)

layer = ReinforcementLayer(z=450, A_s=2000)

# Test at yield
eps_yield = layer.eps_sy
force_yield = layer.get_force(eps_yield)
sig_yield = layer.get_sig(eps_yield)

print(f"\n1. Force at yield:")
print(f"   Yield strain:  {eps_yield:.5f}")
print(f"   Yield stress:  {sig_yield:.1f} MPa")
print(f"   Force:         {force_yield:,.0f} N")
print(f"   Expected:      {layer.A_s * sig_yield:,.0f} N (A_s × σ)")
print(f"   Match:         {np.isclose(force_yield, layer.A_s * sig_yield)}")

# Test moment about different reference points
y_ref_top = 0.0
y_ref_centroid = 250.0

moment_top = layer.get_moment(eps_yield, y_ref_top)
moment_centroid = layer.get_moment(eps_yield, y_ref_centroid)

print(f"\n2. Moment about reference points:")
print(f"   Layer at z = {layer.z:.1f} mm")
print(f"   Force = {force_yield:,.0f} N")
print(f"   ")
print(f"   About top (y=0):      M = {moment_top:,.0f} Nmm")
print(f"   Expected:             M = {force_yield * layer.z:,.0f} Nmm")
print(f"   ")
print(f"   About centroid (y=250): M = {moment_centroid:,.0f} Nmm")
print(f"   Expected:               M = {force_yield * (layer.z - 250):,.0f} Nmm")

print(f"\n✓ Force and moment calculations correct!")

## Test 4: ReinforcementLayout - Multiple Layers

Create a layout with multiple layers and test combined behavior.

In [ ]:
# Test 4: Multiple layers
print("\n" + "=" * 70)
print("TEST 4: ReinforcementLayout - Multiple Layers")
print("=" * 70)

# Create layout and add layers
layout = ReinforcementLayout()

# Top layer
layout.add_layer(z=50, A_s=800, name="Top")

# Middle layers
layout.add_layer(z=250, A_s=600, name="Middle")

# Bottom layer
layout.add_layer(z=450, A_s=1600, name="Bottom")

print(f"\n1. Layout properties:")
print(f"   Number of layers:  {layout.n_layers}")
print(f"   Total steel area:  {layout.total_area:,.0f} mm²")
print(f"   Top position:      {layout.z_min:.1f} mm")
print(f"   Bottom position:   {layout.z_max:.1f} mm")
print(f"   Centroid:          {layout.centroid_z:.1f} mm")

# Verify centroid calculation
sum_Az = sum(layer.A_s * layer.z for layer in layout.layers)
centroid_expected = sum_Az / layout.total_area

print(f"\n2. Centroid verification:")
print(f"   Calculated: {layout.centroid_z:.1f} mm")
print(f"   Expected:   {centroid_expected:.1f} mm")
print(f"   Match:      {np.isclose(layout.centroid_z, centroid_expected)}")

# Layer details
print(f"\n3. Layer details:")
for i, layer in enumerate(layout.layers):
    print(f"   Layer {i+1} ({layer.name}):")
    print(f"      z = {layer.z:.1f} mm, A_s = {layer.A_s:.0f} mm², f_sy = {layer.f_sy:.0f} MPa")

print(f"\n✓ Layout management working correctly!")

## Test 5: Strain Distribution

Test plane section assumption: ε(z) = ε_top - κ×z

In [ ]:
# Test 5: Strain distribution
print("\n" + "=" * 70)
print("TEST 5: Strain Distribution - Plane Sections")
print("=" * 70)

layout = ReinforcementLayout()
layout.add_layer(z=50, A_s=1000, name="Top")
layout.add_layer(z=250, A_s=800, name="Middle")
layout.add_layer(z=450, A_s=2000, name="Bottom")

# Define curvature and top strain
kappa = 0.00001  # 1/mm
eps_top = -0.001  # Compression at top

# Get strain distribution
strains = layout.get_strain_distribution(kappa, eps_top)

print(f"\n1. Strain distribution:")
print(f"   Curvature κ:    {kappa:.6f} 1/mm")
print(f"   Top strain ε₀:  {eps_top:.6f}")
print(f"")
print(f"   {'Layer':<10} {'z [mm]':<10} {'ε [-]':<15} {'Expected':<15} {'Match'}")
print(f"   {'-'*60}")

for i, (layer, eps) in enumerate(zip(layout.layers, strains)):
    eps_expected = eps_top - kappa * layer.z
    match = "✓" if np.isclose(eps, eps_expected) else "✗"
    print(f"   {layer.name:<10} {layer.z:<10.1f} {eps:<15.6f} {eps_expected:<15.6f} {match}")

# Check neutral axis location
z_neutral = -eps_top / kappa if kappa != 0 else float('inf')
print(f"\n2. Neutral axis:")
print(f"   Location: z = {z_neutral:.1f} mm")
print(f"   (where strain = 0)")

# Identify compression/tension zones
print(f"\n3. Stress state:")
for layer, eps in zip(layout.layers, strains):
    if eps < 0:
        state = "Compression"
    elif eps > 0:
        state = "Tension"
    else:
        state = "Neutral"
    print(f"   {layer.name}: {state} (ε = {eps:.6f})")

print(f"\n✓ Plane section assumption correctly implemented!")

## Test 6: Total Force and Moment

Test combined N and M computation for all layers.

In [ ]:
# Test 6: N and M computation
print("\n" + "=" * 70)
print("TEST 6: Total Force and Moment")
print("=" * 70)

layout = ReinforcementLayout()
layout.add_layer(z=50, A_s=1000, name="Top")
layout.add_layer(z=450, A_s=2000, name="Bottom")

# Case 1: Pure compression (negative curvature, top compressed)
kappa1 = -0.000005  # Negative curvature
eps_top1 = -0.003

N1, M1 = layout.get_N_M(kappa1, eps_top1, y_ref=0)

print(f"\n1. Case: Negative curvature (top compressed):")
print(f"   κ = {kappa1:.6f} 1/mm")
print(f"   ε_top = {eps_top1:.6f}")
print(f"   N = {N1:,.0f} N")
print(f"   M = {M1:,.0f} Nmm (about top)")

# Verify manually
strains1 = layout.get_strain_distribution(kappa1, eps_top1)
N_manual = 0
M_manual = 0
for layer, eps in zip(layout.layers, strains1):
    force = layer.get_force(eps)
    N_manual += force
    M_manual += force * layer.z
    print(f"   {layer.name}: ε={eps:.6f}, F={force:,.0f} N")

print(f"   Verification: N={N_manual:,.0f} N, M={M_manual:,.0f} Nmm")
print(f"   Match: {np.isclose(N1, N_manual) and np.isclose(M1, M_manual)}")

# Case 2: Positive curvature (top in tension)
kappa2 = 0.00001
eps_top2 = 0.002

N2, M2 = layout.get_N_M(kappa2, eps_top2, y_ref=0)

print(f"\n2. Case: Positive curvature (top in tension):")
print(f"   κ = {kappa2:.6f} 1/mm")
print(f"   ε_top = {eps_top2:.6f}")
print(f"   N = {N2:,.0f} N")
print(f"   M = {M2:,.0f} Nmm (about top)")

# Moment about different reference point
M2_centroid = layout.get_N_M(kappa2, eps_top2, y_ref=250)[1]
print(f"   M = {M2_centroid:,.0f} Nmm (about y=250)")

print(f"\n✓ Force and moment computation working correctly!")

## Test 7: Helper Functions - Symmetric Reinforcement

Test the convenience function for creating symmetric layouts.

In [ ]:
# Test 7: Symmetric reinforcement
print("\n" + "=" * 70)
print("TEST 7: Helper Functions - Symmetric Reinforcement")
print("=" * 70)

# Create typical beam reinforcement
h_beam = 500  # mm
cover = 40    # mm

layout = create_symmetric_reinforcement(
    h=h_beam,
    cover=cover,
    A_s_top=800,
    A_s_bottom=1600
)

print(f"\n1. Symmetric reinforcement layout:")
print(f"   Beam height:       {h_beam} mm")
print(f"   Cover:             {cover} mm")
print(f"   Number of layers:  {layout.n_layers}")
print(f"   Total steel area:  {layout.total_area:,.0f} mm²")

print(f"\n2. Layer positions:")
for layer in layout.layers:
    print(f"   {layer.name:<10} z = {layer.z:.1f} mm, A_s = {layer.A_s:.0f} mm²")

# Verify positions
top_layer = layout.layers[0]
bottom_layer = layout.layers[1]

print(f"\n3. Position verification:")
print(f"   Top layer:    z = {top_layer.z:.1f} mm (expected: {cover:.1f} mm)")
print(f"   Bottom layer: z = {bottom_layer.z:.1f} mm (expected: {h_beam - cover:.1f} mm)")
print(f"   Match: {np.isclose(top_layer.z, cover) and np.isclose(bottom_layer.z, h_beam - cover)}")

print(f"\n✓ Symmetric reinforcement helper working!")

## Test 8: Helper Functions - Distributed Reinforcement

Test the function for creating uniformly distributed layers.

In [ ]:
# Test 8: Distributed reinforcement
print("\n" + "=" * 70)
print("TEST 8: Helper Functions - Distributed Reinforcement")
print("=" * 70)

# Create wall reinforcement (multiple layers)
h_wall = 3000  # mm
cover = 50     # mm
n_layers = 5
A_s_total = 5000  # mm²

layout = create_distributed_reinforcement(
    h=h_wall,
    cover=cover,
    n_layers=n_layers,
    A_s_total=A_s_total
)

print(f"\n1. Distributed reinforcement layout:")
print(f"   Wall height:       {h_wall} mm")
print(f"   Cover:             {cover} mm")
print(f"   Number of layers:  {layout.n_layers}")
print(f"   Total steel area:  {layout.total_area:,.0f} mm²")
print(f"   Area per layer:    {A_s_total/n_layers:.0f} mm²")

print(f"\n2. Layer distribution:")
print(f"   {'Layer':<12} {'z [mm]':<12} {'A_s [mm²]':<12} {'Spacing [mm]'}")
print(f"   {'-'*50}")

for i, layer in enumerate(layout.layers):
    if i > 0:
        spacing = layer.z - layout.layers[i-1].z
    else:
        spacing = 0
    print(f"   {layer.name:<12} {layer.z:<12.1f} {layer.A_s:<12.0f} {spacing:.1f}")

# Verify uniform spacing
spacings = [layout.layers[i+1].z - layout.layers[i].z 
            for i in range(len(layout.layers)-1)]
uniform = all(np.isclose(s, spacings[0]) for s in spacings)

print(f"\n3. Spacing verification:")
print(f"   All spacings equal: {uniform}")
print(f"   Average spacing:    {np.mean(spacings):.1f} mm")

# Verify total area
print(f"\n4. Area verification:")
print(f"   Total area:    {layout.total_area:.0f} mm²")
print(f"   Expected:      {A_s_total:.0f} mm²")
print(f"   Match:         {np.isclose(layout.total_area, A_s_total)}")

print(f"\n✓ Distributed reinforcement helper working!")

## Test 9: Material Integration - Different Steel Grades

Test that different steel grades work correctly in the same layout.

In [ ]:
# Test 9: Mixed steel grades
print("\n" + "=" * 70)
print("TEST 9: Material Integration - Different Steel Grades")
print("=" * 70)

# Create layout with different steel grades
layout = ReinforcementLayout()

# Top: B500A (lower ductility)
steel_a = create_steel('B500A')
layout.add_layer(z=50, A_s=1000, material=steel_a, name="Top (B500A)")

# Middle: B500B (medium ductility)
steel_b = create_steel('B500B')
layout.add_layer(z=250, A_s=800, material=steel_b, name="Middle (B500B)")

# Bottom: B500C (high ductility)
steel_c = create_steel('B500C')
layout.add_layer(z=450, A_s=1600, material=steel_c, name="Bottom (B500C)")

print(f"\n1. Mixed steel grade layout:")
print(f"   {'Layer':<15} {'f_sy [MPa]':<12} {'f_st [MPa]':<12} {'k ratio'}")
print(f"   {'-'*55}")

for layer in layout.layers:
    k_ratio = layer.material.ductility_ratio
    print(f"   {layer.name:<15} {layer.f_sy:<12.1f} {layer.f_st:<12.1f} {k_ratio:.3f}")

# Test behavior at high strain
eps_large = 0.05  # 5% strain

print(f"\n2. Response at large strain (ε = {eps_large:.3f}):")
print(f"   {'Layer':<15} {'σ [MPa]':<12} {'Behavior'}")
print(f"   {'-'*40}")

for layer in layout.layers:
    sig = layer.get_sig(eps_large)
    
    if eps_large > layer.material.eps_ud:
        behavior = "Failed"
    elif eps_large > layer.eps_sy:
        behavior = "Yielded"
    else:
        behavior = "Elastic"
    
    print(f"   {layer.name:<15} {sig:<12.1f} {behavior}")

# Compute total force
kappa = 0.0  # Uniform strain
eps_top = 0.01
N, M = layout.get_N_M(kappa, eps_top)

print(f"\n3. Combined response (uniform ε = {eps_top:.3f}):")
print(f"   Total force: N = {N:,.0f} N")

print(f"\n✓ Mixed steel grades working correctly!")

## Test 10: Visualization - Reinforcement Layout

Visualize a reinforcement layout in the context of a cross-section.

In [ ]:
# Test 10: Visualization
print("\n" + "=" * 70)
print("TEST 10: Visualization - Reinforcement Layout")
print("=" * 70)

def plot_reinforcement_layout(layout, h, b, ax):
    """Plot reinforcement layout in cross-section."""
    
    # Draw cross-section outline
    from matplotlib.patches import Rectangle
    rect = Rectangle((0, 0), b, h, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    
    # Plot reinforcement layers
    for layer in layout.layers:
        # Bar representation (red circles)
        bar_diameter = np.sqrt(layer.A_s / np.pi) / 5  # Scaled for visibility
        circle = plt.Circle((b/2, layer.z), bar_diameter, 
                          color='red', alpha=0.7, zorder=10)
        ax.add_patch(circle)
        
        # Layer label
        ax.text(b + 20, layer.z, 
               f"{layer.name}\nz={layer.z:.0f}mm\nA={layer.A_s:.0f}mm²",
               verticalalignment='center', fontsize=8)
        
        # Dimension line
        ax.plot([0, b], [layer.z, layer.z], 'r--', alpha=0.3, linewidth=0.5)
    
    # Mark centroid
    centroid_z = layout.centroid_z
    ax.plot([0, b], [centroid_z, centroid_z], 'g--', linewidth=1.5, 
           label=f'Centroid (z={centroid_z:.0f}mm)')
    ax.plot(b/2, centroid_z, 'go', markersize=8)
    
    ax.set_xlim(-50, b + 150)
    ax.set_ylim(-50, h + 50)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Width [mm]')
    ax.set_ylabel('Depth from top [mm]')
    ax.legend(loc='upper left')

# Create example layouts
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Layout 1: Symmetric (beam)
layout1 = create_symmetric_reinforcement(h=500, cover=40, A_s_top=800, A_s_bottom=1600)
plot_reinforcement_layout(layout1, h=500, b=300, ax=axes[0])
axes[0].set_title('Symmetric Reinforcement\n(Typical Beam)', fontsize=12, fontweight='bold')

# Layout 2: Distributed (wall)
layout2 = create_distributed_reinforcement(h=600, cover=50, n_layers=4, A_s_total=3200)
plot_reinforcement_layout(layout2, h=600, b=250, ax=axes[1])
axes[1].set_title('Distributed Reinforcement\n(Wall)', fontsize=12, fontweight='bold')

# Layout 3: Custom (column)
layout3 = ReinforcementLayout()
# Corner bars
for z_pos in [60, 540]:
    layout3.add_layer(z=z_pos, A_s=600, name=f"z={z_pos}")
# Middle bars
for z_pos in [180, 300, 420]:
    layout3.add_layer(z=z_pos, A_s=400, name=f"z={z_pos}")
plot_reinforcement_layout(layout3, h=600, b=400, ax=axes[2])
axes[2].set_title('Custom Layout\n(Column)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Reinforcement layouts visualized successfully!")

## Summary: Day 6 - Reinforcement Layers

### Implemented Components

1. **ReinforcementLayer**: Single reinforcement layer with material integration
2. **ReinforcementLayout**: Collection of layers with analysis methods
3. **Helper Functions**: `create_symmetric_reinforcement()`, `create_distributed_reinforcement()`

### Key Features

✅ **Material Integration**: Uses `SteelReinforcement` from Phase 1  
✅ **Force Computation**: `get_force(eps)` delegates to material model  
✅ **Moment Computation**: `get_moment(eps, y_ref)` with flexible reference point  
✅ **Plane Sections**: `get_strain_distribution(kappa, eps_top)` implements linear strain  
✅ **Combined Analysis**: `get_N_M(kappa, eps_top)` for all layers  
✅ **Layout Management**: Add, remove, query layers  
✅ **Geometric Properties**: Centroid, bounds, total area  

### Testing Coverage

- ✅ Basic properties and material integration
- ✅ Stress-strain delegation to material model
- ✅ Force and moment computation
- ✅ Multiple layer management
- ✅ Strain distribution (plane sections remain plane)
- ✅ Combined N and M computation
- ✅ Helper functions for common patterns
- ✅ Mixed steel grades in same layout
- ✅ Visualization of layouts

### API Usage

```python
# Single layer
layer = ReinforcementLayer(z=50, A_s=1000)
force = layer.get_force(eps=0.002)

# Multiple layers
layout = ReinforcementLayout()
layout.add_layer(z=50, A_s=800, name="Top")
layout.add_layer(z=450, A_s=1600, name="Bottom")

# Analysis
N, M = layout.get_N_M(kappa=0.00001, eps_top=-0.002)

# Helper functions
layout = create_symmetric_reinforcement(
    h=500, cover=40,
    A_s_top=800, A_s_bottom=1600
)
```

### Integration with Phase 1 (matmod)

✅ **Uses SteelReinforcement**: Direct material model integration  
✅ **Uses create_steel()**: Factory for standard grades  
✅ **Delegates stress computation**: `layer.get_sig()` → `material.get_sig()`  
✅ **Inherits properties**: `f_sy`, `E_s`, `eps_sy` from material  

### Next Steps: Phase 2, Step 3

**CrossSection** - Complete assembly:
- Combine geometric shape + concrete material + reinforcement
- Implement `get_N_M(kappa, eps_top)` that integrates concrete and steel
- This is the KEY METHOD that mkappa will use in Phase 3
- Visualization of complete cross-section with strains and stresses

**Location**: `bmcs_cross_section/cs_design/cross_section.py`

---

**Phase 2 Step 2 Complete!** ✅